In [3]:
# ============================================================
# PACKAGE 10.2
# WARD-LEVEL GEOSPATIAL SOURCE ATTRIBUTION ENGINE
# ============================================================


import geopandas as gpd
import pandas as pd
import numpy as np
import os
import json
import folium


print("="*75)
print("PACKAGE 10.2 : WARD LEVEL GEOSPATIAL SOURCE ATTRIBUTION")
print("="*75)



# ============================================================
# PATH CONFIGURATION
# ============================================================


BASE_PATH = r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE"


WARD_FILE = os.path.join(
    BASE_PATH,
    "Geospatial_Pollution_Source_Attribution_Engine",
    "Ward_Level_Source_ Attribution",
    "Delhi_Wards.geojson"
)


ROAD_FILE = os.path.join(
    BASE_PATH,
    "Cleaning",
    "Road_Networks",
    "Delhi_Road_Network_cleaned.geojson"
)


CONSTRUCTION_FILE = os.path.join(
    BASE_PATH,
    "Cleaning",
    "cleaned_construction_waste.geojson"
)


INDUSTRIAL_FILE = os.path.join(
    BASE_PATH,
    "Cleaning",
    "cleaned_industrial.geojson"
)


SENSITIVE_FILE = os.path.join(
    BASE_PATH,
    "Cleaning",
    "cleaned_sensitive_locations.geojson"
)


OUTPUT_FOLDER = os.path.join(
    BASE_PATH,
    "Geospatial_Pollution_Source_Attribution_Engine",
    "Ward_Level_Source_Attribution_Outputs"
)


os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)



# ============================================================
# LOAD DATA
# ============================================================


print("\nLoading Spatial Datasets...")


wards = gpd.read_file(WARD_FILE)

roads = gpd.read_file(ROAD_FILE)

construction = gpd.read_file(CONSTRUCTION_FILE)

industrial = gpd.read_file(INDUSTRIAL_FILE)

sensitive = gpd.read_file(SENSITIVE_FILE)



print("\nDataset Loaded")

print("Wards:", wards.shape)

print("Roads:", roads.shape)

print("Construction:", construction.shape)

print("Industrial:", industrial.shape)

print("Sensitive:", sensitive.shape)



# ============================================================
# CRS STANDARDIZATION
# ============================================================


datasets = [
    wards,
    roads,
    construction,
    industrial,
    sensitive
]


for df in datasets:

    if df.crs != "EPSG:4326":

        df.to_crs(
            "EPSG:4326",
            inplace=True
        )



# ============================================================
# SPATIAL JOIN FUNCTION
# ============================================================


def count_features(
        wards,
        features,
        name
):

    joined = gpd.sjoin(
        features,
        wards[
            [
                "Ward_Name",
                "Ward_No",
                "geometry"
            ]
        ],
        predicate="intersects",
        how="left"
    )


    counts = (
        joined
        .groupby("Ward_No")
        .size()
        .reset_index(
            name=name
        )
    )


    return counts




print("\nPerforming Spatial Attribution...")



# ============================================================
# SOURCE COUNTS
# ============================================================


road_count = count_features(
    wards,
    roads,
    "Traffic_Feature_Count"
)


construction_count = count_features(
    wards,
    construction,
    "Construction_Count"
)


industrial_count = count_features(
    wards,
    industrial,
    "Industrial_Count"
)


sensitive_count = count_features(
    wards,
    sensitive,
    "Sensitive_Count"
)



# ============================================================
# MERGE RESULTS
# ============================================================


ward_data = wards[
    [
        "Ward_Name",
        "Ward_No",
        "geometry"
    ]
].copy()



for df in [

    road_count,

    construction_count,

    industrial_count,

    sensitive_count

]:

    ward_data = ward_data.merge(
        df,
        on="Ward_No",
        how="left"
    )



ward_data.fillna(
    0,
    inplace=True
)



# ============================================================
# NORMALIZATION
# ============================================================


def normalize(col):

    if ward_data[col].max()==0:

        return np.zeros(
            len(ward_data)
        )

    return (
        ward_data[col] /
        ward_data[col].max()
    )



ward_data[
"Traffic_Score"
]=normalize(
"Traffic_Feature_Count"
)



ward_data[
"Construction_Score"
]=normalize(
"Construction_Count"
)



ward_data[
"Industrial_Score"
]=normalize(
"Industrial_Count"
)



ward_data[
"Sensitivity_Score"
]=normalize(
"Sensitive_Count"
)



# ============================================================
# POLLUTION SOURCE SCORE
# ============================================================


ward_data["Traffic_%"] = (
    ward_data["Traffic_Score"]
    /
    (
        ward_data[
        [
        "Traffic_Score",
        "Construction_Score",
        "Industrial_Score"
        ]
        ].sum(axis=1)
        +0.001
    )
)*100



ward_data["Construction_%"] = (
    ward_data["Construction_Score"]
    /
    (
        ward_data[
        [
        "Traffic_Score",
        "Construction_Score",
        "Industrial_Score"
        ]
        ].sum(axis=1)
        +0.001
    )
)*100



ward_data["Industrial_%"] = (
    ward_data["Industrial_Score"]
    /
    (
        ward_data[
        [
        "Traffic_Score",
        "Construction_Score",
        "Industrial_Score"
        ]
        ].sum(axis=1)
        +0.001
    )
)*100



# ============================================================
# DOMINANT SOURCE
# ============================================================


source_columns = [

"Traffic_%",

"Construction_%",

"Industrial_%"

]


mapping = {

"Traffic_%":
"Traffic Emission",

"Construction_%":
"Construction Activity",

"Industrial_%":
"Industrial Pollution"

}



ward_data["Dominant_Source"] = (

ward_data[source_columns]
.idxmax(axis=1)

.map(mapping)

)



ward_data["Confidence"] = (

ward_data[source_columns]
.max(axis=1)

.apply(

lambda x:

"High"

if x>50

else

"Medium"

if x>30

else

"Low"

)

)



# ============================================================
# SAVE CSV
# ============================================================


csv_output = ward_data.drop(
    columns="geometry"
)


csv_output.to_csv(

os.path.join(
OUTPUT_FOLDER,
"Ward_Source_Attribution.csv"
),

index=False

)



confidence = csv_output[
[
"Ward_Name",
"Dominant_Source",
"Confidence"
]
]


confidence.to_csv(

os.path.join(
OUTPUT_FOLDER,
"Ward_Source_Confidence.csv"
),

index=False

)



# ============================================================
# HOTSPOTS
# ============================================================


hotspots = csv_output.sort_values(

by=[
"Traffic_%",
"Industrial_%",
"Construction_%"
],

ascending=False

).head(20)



hotspots.to_csv(

os.path.join(
OUTPUT_FOLDER,
"Ward_Pollution_Hotspots.csv"
),

index=False

)



# ============================================================
# FRONTEND JSON
# ============================================================


frontend = csv_output.to_dict(
orient="records"
)



with open(

os.path.join(
OUTPUT_FOLDER,
"Frontend_Ward_Attribution.json"
),

"w"

) as f:


    json.dump(
        frontend,
        f,
        indent=4
    )



# ============================================================
# MAP GENERATION
# ============================================================


center = [
28.6139,
77.2090
]


m = folium.Map(
location=center,
zoom_start=11
)



folium.GeoJson(

ward_data,

tooltip=folium.GeoJsonTooltip(

fields=[

"Ward_Name",

"Dominant_Source",

"Confidence"

]

)

).add_to(m)



m.save(

os.path.join(
OUTPUT_FOLDER,
"Ward_Source_Attribution_Map.html"
)

)



# ============================================================
# METADATA
# ============================================================


with open(

os.path.join(
OUTPUT_FOLDER,
"Package10.2_Metadata.txt"
),

"w"

) as f:


    f.write(
"""
PACKAGE 10.2
Ward-Level Geospatial Source Attribution

Purpose:
Identify dominant pollution sources at ward level.

Inputs:
Delhi Ward Boundaries
Road Network
Industrial Zones
Construction Sites
Sensitive Locations


Outputs:
Ward Source Contribution
Ward Confidence Scores
Pollution Hotspots
Frontend JSON
Interactive Map

"""

)



print("\n============================================================")

print("PACKAGE 10.2 COMPLETED SUCCESSFULLY")

print("============================================================")


print("\nFiles Generated:")

for file in os.listdir(OUTPUT_FOLDER):

    print(file)

PACKAGE 10.2 : WARD LEVEL GEOSPATIAL SOURCE ATTRIBUTION

Loading Spatial Datasets...

Dataset Loaded
Wards: (290, 3)
Roads: (24463, 7)
Construction: (135, 7)
Industrial: (95, 5)
Sensitive: (983, 5)

Performing Spatial Attribution...

PACKAGE 10.2 COMPLETED SUCCESSFULLY

Files Generated:
Frontend_Ward_Attribution.json
Package10.2_Metadata.txt
Ward_Pollution_Hotspots.csv
Ward_Source_Attribution.csv
Ward_Source_Attribution_Map.html
Ward_Source_Confidence.csv
